# Claude Code permanente no vlab

O `claude` morre quando o Terminal do Jupyter fecha. Aqui ele roda **dentro do tmux**, que sobrevive a fechar o Terminal ou este notebook, num **laço que o reinicia sozinho** (`claude --continue`, que retoma a conversa desta pasta).

O binário, o login e a configuração ficam em `/home/jovyan/privado/.claude_persist`, que é persistente.

**Limite:** nada sobrevive ao **desligamento do servidor** do Jupyter (*Stop My Server* ou desligamento por ociosidade). Depois de um desligamento, rode as células **1, 2, 4 e 4b** de novo; é rápido e o login é mantido.

| Célula | Quando |
| :--- | :--- |
| 1. Restaurar | sempre primeiro |
| 2. Instalar / reparar | primeira vez e depois de reiniciar o servidor |
| 3. Login (no Terminal) | só na primeira vez |
| 4. Iniciar | depois de 2; se já estiver rodando, não duplica |
| 4b. Sentinela | depois da 4: manda o estado ao Drive a cada 10 min |
| 5. Estado | a qualquer momento |
| 6. Mandar instrução | sem abrir o Terminal |
| 7. Tela | ler a resposta |
| 7b. Tela ao vivo | acompanhar enquanto ele trabalha (só olha) |
| 7c. Histórico | o que ele fez, com os comandos exatos (só lê) |
| 8. Parar | encerra o Claude e o laço |

**Não use *Run All* para tudo:** as células 6 (Mandar) e 8 (Parar) estão protegidas; é preciso tirar o `#` para elas agirem.

## 1. Restaurar o ambiente

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash restaurar.sh


## 2. Instalar / reparar
Instala o tmux (conda, em `conda_envs/ferramentas`) e o Claude Code, se faltarem, e refaz os links para a área persistente. Se o Claude Code já está instalado, só confere.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh instalar


## 3. Login (só na primeira vez, no **Terminal**)
Rode a célula 4 e depois, num Terminal do Jupyter (*File → New → Terminal*):
```
tmux attach -t claude
```
Faça o login (o `claude` mostra um link; abra no navegador do seu computador e cole o código). Na pergunta de confiança, escolha **Yes, I trust this folder**. Para sair **sem encerrar** o Claude: **Ctrl+B** e depois **D**.

O login fica em `/home/jovyan/privado/.claude_persist` e vale para as próximas vezes.

## 4. Iniciar
Cria a sessão tmux `claude`, com reinício automático. Se ela já existe, não duplica.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh iniciar


## 4b. Sentinela (recomendado)
A cada 10 min grava o estado do Claude (sessão e últimas linhas da tela) e da fila do CM1 em `gdrive:vlab_cache/Himalaia_2026/saida/toro_cm1_lhende/ao_vivo/estado_vlab.txt`. No Windows: `G:\Meu Drive\vlab_cache\Himalaia_2026\saida\toro_cm1_lhende\ao_vivo\`. Assim a sessão do Claude no seu computador verifica o vlab sozinha, sem você copiar e colar. Também roda no tmux e sobrevive a fechar o notebook.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh sentinela


## 5. Estado
Mostra se a sessão está viva, o histórico de reinícios e as últimas linhas da tela do Claude.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh estado


## 6. Mandar uma instrução (sem abrir o Terminal)
Troque o texto de `MSG`. Ele é digitado no Claude, seguido de Enter. Para ver a resposta, use a célula 7 depois de alguns segundos ou minutos.

Exemplos:
* `Leia a ordem com rclone cat gdrive:vlab_cache/Himalaia_2026/entrada/ORDEM_reproducao_rad06.md e execute-a. Eu autorizo.`
* `/remote-control` — liga o Remote Control, para a sessão poder ser alcançada de fora, se o seu plano e o contêiner permitirem.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
MSG='Rode bash rodar_direto.sh estado e me diga o resultado.'
# PROTEGIDA contra 'Run All': tire o # da linha abaixo para enviar
# bash claude_permanente.sh mandar "$MSG"
echo 'celula protegida: nada foi enviado (tire o # para mandar)'


## 7. Tela
Últimas linhas da tela do Claude (troque 80 por outro número, se quiser).

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
bash claude_permanente.sh tela 80


## 7b. Tela ao vivo
Mostra a tela do Claude atualizando a cada 5 s, por `MINUTOS` minutos (padrão 10). **Só olha**: nada é digitado no Claude.
O botão Interromper (■) para a exibição; o Claude continua rodando. Num *Run All*, esta célula segura a execução pelos `MINUTOS`.

In [ ]:
import os, shutil, subprocess, time
from IPython.display import clear_output
MINUTOS = 10
TMUX = '/home/jovyan/privado/conda_envs/ferramentas/bin/tmux'
TMUX = TMUX if os.path.exists(TMUX) else (shutil.which('tmux') or TMUX)
try:
    for i in range(MINUTOS * 12):
        s = subprocess.run([TMUX, 'capture-pane', '-p', '-t', 'claude', '-S', '-45'], capture_output=True, text=True)
        clear_output(wait=True)
        print(time.strftime('%H:%M:%S'), '| ao vivo, atualiza a cada 5 s (Interromper para parar)\n')
        print(s.stdout or s.stderr or 'sessao claude nao encontrada (rode as celulas 1, 2 e 4)')
        time.sleep(5)
except KeyboardInterrupt:
    print('exibicao parada (o Claude continua rodando)')


## 7c. O que o Claude fez (histórico com os comandos)
Lê o histórico da conversa que o próprio Claude Code grava e mostra os últimos `N` eventos:
* **VOCE**: as suas mensagens;
* **CLAUDE**: as respostas dele;
* **COMANDO**: cada comando que ele rodou, exatamente como rodou;
* **RESULTADO**: o fim da saída de cada comando.

Os horários estão em UTC. **Só lê**: não muda nada.

In [ ]:
import json, glob, os
N = 40                      # quantos eventos mostrar (os ultimos)
P = '/home/jovyan/privado/.claude_persist/.claude/projects'
arqs = glob.glob(P + '/*/*.jsonl')
if not arqs:
    print('nenhum historico em', P)
else:
    f = max(arqs, key=os.path.getmtime)
    ev = []
    for l in open(f, encoding='utf-8'):
        try: d = json.loads(l)
        except Exception: continue
        c = (d.get('message') or {}).get('content'); t = d.get('timestamp', '')[11:19] + ' UTC'
        if isinstance(c, str):
            ev.append((t, 'VOCE' if d.get('type') == 'user' else 'CLAUDE', c)); continue
        for b in c or []:
            if not isinstance(b, dict): continue
            k = b.get('type')
            if k == 'text': ev.append((t, 'CLAUDE', b.get('text', '')))
            elif k == 'tool_use':
                i = b.get('input') or {}
                ev.append((t, 'COMANDO', i.get('command') or json.dumps(i, ensure_ascii=False)[:400]))
            elif k == 'tool_result':
                r = b.get('content'); r = r if isinstance(r, str) else ' '.join(x.get('text', '') for x in (r or []) if isinstance(x, dict))
                ev.append((t, 'RESULTADO', r[-700:]))
    print('historico:', f, '| eventos:', len(ev), '\n')
    for t, k, x in ev[-N:]:
        print(f'[{t}] {k}:\n  ' + x.strip()[:1500].replace('\n', '\n  ') + '\n')


## 8. Parar
Encerra a sessão tmux e o laço de reinício. As rodadas do CM1 lançadas pelo `rodar_direto.sh` **continuam**; elas não dependem do Claude.

Para parar só a sentinela: `bash claude_permanente.sh parar_sentinela`.

In [ ]:
%%bash -l
cd /home/jovyan/privado/Himalaia_2026/toro_cm1_lhende
# PROTEGIDA contra 'Run All': tire o # da linha abaixo para encerrar o Claude de proposito
# bash claude_permanente.sh parar
echo 'celula protegida: nada foi encerrado (tire o # para parar)'
